# Competition Analysis — AI Actors Network

Analyse concurrentielle à partir des données de la base SQLite :
1. Export des couples entreprise / concurrent
2. Format long + agrégation
3. Matrice de cooccurrence
4. Projection 2D

## 1. Configuration et imports

In [1]:
import sqlite3
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.preprocessing import normalize

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

COL_NAME        = "name"
COL_SECTOR      = "sector"
COL_COMPETITORS = "main_competitors"
RANDOM_SEED     = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

Base : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. Extraction SQL — entreprise / secteurs / concurrents

In [2]:
TOP_N = 100   # nombre d'entreprises retenues pour l'analyse

# ranking_score = MAX(capitalization, funds_raised) en numérique — même logique que server.js
SQL = f"""
SELECT
    name             AS {COL_NAME},
    sector           AS {COL_SECTOR},
    main_competitors AS {COL_COMPETITORS},
    MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
    ) AS ranking_score
FROM enterprises
WHERE main_competitors IS NOT NULL
  AND main_competitors != ''
  AND MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
      ) > 0
ORDER BY ranking_score DESC
LIMIT {TOP_N}
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(f"{len(df_raw)} entreprises (top {TOP_N} par capitalisation / fonds levés)")
df_raw[["name", "ranking_score"]].head(10)

59 entreprises (top 100 par capitalisation / fonds levés)


,name,ranking_score
0,ByteDance,50000.0
1,Alibaba,30000.0
2,Scale AI,16000.0
3,Cerebras,8500.0
4,Wayve,2500.0
5,UiPath,1960.0
6,Celonis,1770.0
7,YouTube,1680.0
8,Cohere,1600.0
9,World Labs,1230.0


In [27]:
# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

Export brut → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Nettoyage et standardisation des noms de concurrents

In [3]:
_SEP = re.compile(r"[,;/\n]+")
_INVALID = re.compile(r"^(na|n/a|none|unknown|tbd|-)$", re.IGNORECASE)

def clean_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = re.sub(r"\s{2,}", " ", s)
    return s.title()

def split_competitors(raw: str) -> list[str]:
    parts = _SEP.split(str(raw))
    return [clean_name(p) for p in parts
            if clean_name(p) and not _INVALID.match(p.strip())]

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != r[COL_NAME].lower()],
    axis=1,
)

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises après nettoyage")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

59 entreprises après nettoyage


,name,main_competitors
0,ByteDance,"[Meta, Youtube, Tencent, Openai, Google Deepmind]"
1,Alibaba,"[Pinduoduo, Jd.Com, Bytedance, Temu, Shein, Am..."
2,Scale AI,"[Srge Ai, Snorkel Ai, Data Annotation, Appen, ..."
3,Cerebras,"[Nvidia, Amd, Intel, Google . Amazon Web Servi..."
4,Wayve,[Waymo]
5,UiPath,"[N, A]"
6,Celonis,"[Sap Signavio, Microsoft Power Automate, Uipat..."
7,YouTube,"[Tiktok, Instagram, Twitch, Vimeo, Deezer]"


## 3b. Normalisation sémantique — filiales → groupe parent

In [5]:
SEMANTIC_ALIASES: dict[str, str] = {
    # ── Google (nom canonique dans la base) ───────────────────────────────────
    "Alphabet":             "Google",
    "Alphabet Inc.":        "Google",
    "Youtube":              "Google",
    "YouTube":              "Google",
    "Deepmind":             "Google",
    "DeepMind":             "Google",
    "Google Deepmind":      "Google",
    "Google DeepMind":      "Google",
    "Google Brain":         "Google",
    "Google Cloud":         "Google",
    "Google Cloud Platform":"Google",
    "GCP":                  "Google",
    "Waymo":                "Google",
    "Verily":               "Google",
    "Calico":               "Google",
    "Waze":                 "Google",
    "Google Translate":     "Google",
    "Gmail":                "Google",
    "Android":              "Google",
    # ── Meta (nom canonique dans la base) ─────────────────────────────────────
    "Meta Platforms":       "Meta",
    "Facebook":             "Meta",
    "Instagram":            "Meta",
    "Whatsapp":             "Meta",
    "WhatsApp":             "Meta",
    "Threads":              "Meta",
    "Oculus":               "Meta",
    "Meta Quest":           "Meta",
    "LLaMA":                "Meta",
    "Llama":                "Meta",
    # ── Microsoft ─────────────────────────────────────────────────────────────
    "Azure":                "Microsoft",
    "Microsoft Azure":      "Microsoft",
    "Linkedin":             "Microsoft",
    "LinkedIn":             "Microsoft",
    "Github":               "Microsoft",
    "GitHub":               "Microsoft",
    "Skype":                "Microsoft",
    "Bing":                 "Microsoft",
    "Nuance":               "Microsoft",
    "Nuance Communications":"Microsoft",
    "Activision Blizzard":  "Microsoft",
    "Activision":           "Microsoft",
    "Xbox":                 "Microsoft",
    "Microsoft Translator": "Microsoft",
    "Office 365":           "Microsoft",
    # ── Amazon ────────────────────────────────────────────────────────────────
    "Aws":                  "Amazon",
    "AWS":                  "Amazon",
    "Alexa":                "Amazon",
    "Twitch":               "Amazon",
    "Amazon.Com":           "Amazon",
    "Amazon Prime":         "Amazon",
    "Amazon Prime Video":   "Amazon",
    "Kindle":               "Amazon",
    # ── Amazon Web Services (entité séparée dans la base) ─────────────────────
    "Amazon Web Services (AWS)": "Amazon Web Services",
    # ── Apple ─────────────────────────────────────────────────────────────────
    "Siri":                 "Apple",
    "Apple Inc.":           "Apple",
    "Apple Inc":            "Apple",
    "Iphone":               "Apple",
    "iPhone":               "Apple",
    "Ipad":                 "Apple",
    "iPad":                 "Apple",
    "Apple Silicon":        "Apple",
    # ── Salesforce ────────────────────────────────────────────────────────────
    "Slack":                "Salesforce",
    "Tableau":              "Salesforce",
    "Mulesoft":             "Salesforce",
    "MuleSoft":             "Salesforce",
    "Salesforce - Einstein":"Salesforce",
    "Einstein":             "Salesforce",
    # ── IBM ───────────────────────────────────────────────────────────────────
    "Red Hat":              "IBM",
    "RedHat":               "IBM",
    "Watsonx":              "IBM",
    "Watson":               "IBM",
    "IBM Watson":           "IBM",
    # ── Oracle ────────────────────────────────────────────────────────────────
    "Netsuite":             "Oracle",
    "NetSuite":             "Oracle",
    "Java":                 "Oracle",
    # ── Nvidia (nom canonique dans la base) ───────────────────────────────────
    "NVIDIA":               "Nvidia",
    "Cuda":                 "Nvidia",
    "CUDA":                 "Nvidia",
    "Nvidia Corporation":   "Nvidia",
    # ── INtel (nom avec cette casse dans la base) ─────────────────────────────
    "Intel":                "INtel",
    "Intel Corporation":    "INtel",
    # ── AMD ───────────────────────────────────────────────────────────────────
    "AMD Inc.":             "AMD",
    "Advanced Micro Devices": "AMD",
    # ── ByteDance ─────────────────────────────────────────────────────────────
    "Tiktok":               "ByteDance",
    "TikTok":               "ByteDance",
    "Douyin":               "ByteDance",
    "Bytedance":            "ByteDance",
    # ── X (nom canonique dans la base, anciennement Twitter) ──────────────────
    "Twitter":              "X",
    "X.Com":                "X",
    "X Corp":               "X",
    # ── Tesla ─────────────────────────────────────────────────────────────────
    "Tesla Inc.":           "Tesla",
    "Tesla Motors":         "Tesla",
    # ── SpaceX ────────────────────────────────────────────────────────────────
    "Space Exploration Technologies": "SpaceX",
    "Starlink":             "SpaceX",
    # ── Baidu ─────────────────────────────────────────────────────────────────
    "Ernie":                "Baidu",
    "Ernie Bot":            "Baidu",
    "ERNIE":                "Baidu",
    # ── Tencent ───────────────────────────────────────────────────────────────
    "Wechat":               "Tencent",
    "WeChat":               "Tencent",
    "Qq":                   "Tencent",
    "QQ":                   "Tencent",
    # ── OpenAI ────────────────────────────────────────────────────────────────
    "Chatgpt":              "OpenAI",
    "ChatGPT":              "OpenAI",
    "Gpt-4":                "OpenAI",
    "GPT-4":                "OpenAI",
    "Gpt4":                 "OpenAI",
    "GPT4":                 "OpenAI",
    "Openai":               "OpenAI",
    "DALL-E":               "OpenAI",
    "Dall-E":               "OpenAI",
    "Sora":                 "OpenAI",
    # ── Anthropic ─────────────────────────────────────────────────────────────
    "Claude":               "Anthropic",
    "Claude AI":            "Anthropic",
    # ── Samsung ───────────────────────────────────────────────────────────────
    "Samsung Electronics":  "Samsung",
    "Samsung System LSI":   "Samsung",
    # ── Adobe ─────────────────────────────────────────────────────────────────
    "Adobe Firefly":        "Adobe",
    "Photoshop":            "Adobe",
    # ── Qualcomm ──────────────────────────────────────────────────────────────
    "Qualcomm Inc.":        "Qualcomm",
    # ── SAP ───────────────────────────────────────────────────────────────────
    "SAP SE":               "SAP",
    # ── Huawei ────────────────────────────────────────────────────────────────
    "Huawei Technologies":  "Huawei",
    # ── Alibaba ───────────────────────────────────────────────────────────────
    "Alibaba Group":        "Alibaba",
    "AliCloud":             "Alibaba",
    "Alipay":               "Alibaba",
    # ── HP ────────────────────────────────────────────────────────────────────
    "Hewlett-Packard":      "HP",
    "Hewlett Packard":      "HP",
    # ── Netflix ───────────────────────────────────────────────────────────────
    "Netflix Inc.":         "Netflix",
    # ── Spotify ───────────────────────────────────────────────────────────────
    "Spotify AB":           "Spotify",
    # ── Sony ──────────────────────────────────────────────────────────────────
    "Sony Corporation":     "Sony",
    "PlayStation":          "Sony",
    # ── Dell ──────────────────────────────────────────────────────────────────
    "Dell Technologies":    "Dell",
    # ── Lenovo ────────────────────────────────────────────────────────────────
    "Lenovo Group":         "Lenovo",
    # ── MediaTek ──────────────────────────────────────────────────────────────
    "MediaTek Inc.":        "MediaTek",
    # ── Xiaomi ────────────────────────────────────────────────────────────────
    "Xiaomi Corporation":   "Xiaomi",
    # ── Oppo ──────────────────────────────────────────────────────────────────
    "OPPO":                 "Oppo",
    # ── Vivo ──────────────────────────────────────────────────────────────────
    "VIVO":                 "Vivo",
    # ── Disney ────────────────────────────────────────────────────────────────
    "Disney+":              "Disney",
    "Walt Disney":          "Disney",
    # ── Boeing ────────────────────────────────────────────────────────────────
    "Boeing Company":       "Boeing",
    # ── Autres acteurs spécifiques IA ─────────────────────────────────────────
    "Mistral":              "Mistral AI",
    "Stability AI":         "Stability",
    "Stability.ai":         "Stability",
    "Midjourney Inc.":      "Midjourney",
    "Runway ML":            "Runway",
}

def apply_semantic_aliases(names: list[str]) -> list[str]:
    return [SEMANTIC_ALIASES.get(n, n) for n in names]

df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(apply_semantic_aliases)
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(lambda lst: list(dict.fromkeys(lst)))
df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS]
               if c.lower() != r[COL_NAME].lower()
               and SEMANTIC_ALIASES.get(r[COL_NAME].title(), r[COL_NAME]).lower() != c.lower()],
    axis=1,
)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

# explode garde le nom COL_COMPETITORS, pas "competitor"
preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases | {len(df_clean)} entreprises après normalisation sémantique")
print("\nTop concurrents après normalisation :")
print(preview[COL_COMPETITORS].value_counts().head(15).to_string())

143 aliases | 59 entreprises après normalisation sémantique

Top concurrents après normalisation :
main_competitors
Google       16
OpenAI       10
Microsoft    10
Amazon        8
Meta          6
Nvidia        6
Apple         6
ByteDance     5
INtel         5
N             5
Anthropic     5
Tencent       4
A             4
Qualcomm      4
Amd           3


## 4. Format long — couples entreprise–concurrent

In [6]:
df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
)
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entreprise–concurrent")
print(f"Export → {long_path}")
df_long.head(10)

319 couples entreprise–concurrent
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,ByteDance,Meta,Media & Entertainment,50000.0
1,1,ByteDance,Google,Media & Entertainment,50000.0
2,2,ByteDance,Tencent,Media & Entertainment,50000.0
3,3,ByteDance,OpenAI,Media & Entertainment,50000.0
4,4,Alibaba,Pinduoduo,"Cloud Provider, Financial Services, Retail & E...",30000.0
5,5,Alibaba,Jd.Com,"Cloud Provider, Financial Services, Retail & E...",30000.0
6,6,Alibaba,ByteDance,"Cloud Provider, Financial Services, Retail & E...",30000.0
7,7,Alibaba,Temu,"Cloud Provider, Financial Services, Retail & E...",30000.0
8,8,Alibaba,Shein,"Cloud Provider, Financial Services, Retail & E...",30000.0
9,9,Alibaba,Amazon,"Cloud Provider, Financial Services, Retail & E...",30000.0


## 5. Agrégation et comptage des relations concurrentielles

In [7]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entreprise–concurrent :")
display(df_agg.head(20))

print("\nDistribution des fréquences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport → {agg_path}")

Top 20 paires entreprise–concurrent :


,name,competitor,count
0,ByteDance,Meta,1
1,ByteDance,Google,1
2,ByteDance,Tencent,1
3,ByteDance,OpenAI,1
4,Alibaba,Pinduoduo,1
5,Alibaba,Jd.Com,1
6,Alibaba,ByteDance,1
7,Alibaba,Temu,1
8,Alibaba,Shein,1
9,Alibaba,Amazon,1



Distribution des fréquences :


count    319.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 6. Matrice de cooccurrence

In [8]:
# Tableau croisé entreprise × concurrent (asymétrique)
pivot = df_agg.pivot_table(
    index=COL_NAME, columns="competitor", values="count", fill_value=0
)

all_actors = sorted(set(pivot.index) | set(pivot.columns))
pivot = pivot.reindex(index=all_actors, columns=all_actors, fill_value=0)

# Cooccurrence symétrique : M + M^T
cooc = pivot.values + pivot.values.T
np.fill_diagonal(cooc, 0)
df_cooc = pd.DataFrame(cooc, index=all_actors, columns=all_actors)

print(f"Matrice {df_cooc.shape[0]} × {df_cooc.shape[1]} acteurs")
print(f"Densité non-nulle : {(cooc > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_cooc.to_csv(cooc_path)
print(f"Export → {cooc_path}")

Matrice 276 × 276 acteurs
Densité non-nulle : 0.8%
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv


## 7. Projection 2D de la cooccurrence

On choisit la méthode selon la taille : PCA si > 500 acteurs, MDS sinon (MDS préserve mieux les distances de cooccurrence pour des jeux de taille raisonnable).

In [9]:
import warnings

N = len(all_actors)
X = normalize(cooc, norm="l2")

score_map        = df_agg.groupby("competitor")["count"].sum().to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

if N > 500:
    print(f"PCA (N={N} > 500)")
    coords = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X)
    method = "PCA"
else:
    print(f"MDS (N={N} ≤ 500)")
    sim = X @ X.T
    dissim = 1 - np.clip(sim, 0, 1)
    np.fill_diagonal(dissim, 0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        reducer = MDS(n_components=2, dissimilarity="precomputed",
                      init="random", random_state=RANDOM_SEED, normalized_stress="auto")
        coords = reducer.fit_transform(dissim)
    method = "MDS"

df_coords = pd.DataFrame({"actor": all_actors, "x": coords[:, 0], "y": coords[:, 1]})
df_coords["score"]         = df_coords["actor"].map(score_map).fillna(0)
df_coords["log_score"]     = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("External")

coords_path = EXPORTS_DIR / "coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D → {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

MDS (N=276 ≤ 500)
Coordonnées 2D → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv


,actor,x,y,score,log_score,ranking_score,sector
49,ByteDance,-0.303924,-0.019668,5.0,0.778151,50000.0,Media & Entertainment
11,Alibaba,-0.685135,-0.200773,0.0,0.000000,30000.0,Cloud Provider
204,Scale AI,0.356136,-0.648284,0.0,0.000000,16000.0,ICT
53,Cerebras,-0.011722,-0.198542,0.0,0.000000,8500.0,Hardware
264,Wayve,0.093777,-0.222701,0.0,0.000000,2500.0,ICT
252,UiPath,0.359695,-0.388389,0.0,0.000000,1960.0,ICT
52,Celonis,0.412767,-0.662065,0.0,0.000000,1770.0,ICT
274,YouTube,-0.362113,0.131512,0.0,0.000000,1680.0,Media & Entertainment
63,Cohere,-0.248576,0.293167,1.0,0.301030,1600.0,AI model
268,World Labs,0.125600,-0.283548,0.0,0.000000,1230.0,AI model


## 8. Visualisation 2D et export des artefacts

In [10]:
import plotly.express as px

info_cols = ["founded_year", "country", "employees_count",
             "revenue_millions", "capitalization", "funds_raised", "description"]
available = [c for c in info_cols if c in df_raw.columns]
df_info   = df_raw.set_index(COL_NAME)[available]

df_plot = df_coords.copy()
for col in available:
    df_plot[col] = df_plot["actor"].map(df_info[col])

def fmt_hover(row):
    lines = [f"<b>{row['actor']}</b>"]
    if pd.notna(row.get("sector")) and row["sector"] != "External":
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")
    cap = float(row.get("capitalization") or 0)
    if cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")
    rev = float(row.get("revenue_millions") or 0)
    if rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")
    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 160 else ''}</i>")
    lines.append(f"Citations: {int(row['score'])}")
    return "<br>".join(lines)

df_plot["hover"]       = df_plot.apply(fmt_hover, axis=1)
df_plot["marker_size"] = np.maximum(df_plot["score"] * 2, 3)

fig = px.scatter(
    df_plot,
    x="x", y="y",
    color="sector",
    size="marker_size",
    size_max=22,
    text="actor",
    custom_data=["hover"],
    color_discrete_sequence=px.colors.qualitative.Light24,
    title=f"Espace concurrentiel — {method}  ({N} acteurs · taille ∝ citations)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "sector": "Secteur"},
    width=1400,
    height=1200,
)

fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(opacity=0.78, line=dict(width=0.4, color="white")),
)

fig.update_layout(
    legend=dict(title="Secteur", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
)

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive → {fig_html}")
fig.show()

Carte interactive → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [35]:
print("── Récapitulatif des exports ────────────────────────────")
for p in [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]:
    size_kb = Path(p).stat().st_size / 1024
    print(f"  {p.name:<42} {size_kb:6.1f} KB")

── Récapitulatif des exports ────────────────────────────
  competitors_raw.csv                           4.9 KB
  competitors_long.csv                         12.2 KB
  competitors_aggregated.csv                    5.3 KB
  cooccurrence_matrix.csv                     199.3 KB
  coords_2d.csv                                18.7 KB
  competition_map_2d.html                    4783.6 KB
